In [2]:
!pip install faiss-cpu

   ---------------------------------------- 0.0/18.8 MB ? eta -:--:--
   -- ------------------------------------- 1.0/18.8 MB 10.1 MB/s eta 0:00:02
   --- ------------------------------------ 1.8/18.8 MB 5.0 MB/s eta 0:00:04
   ------ --------------------------------- 2.9/18.8 MB 5.6 MB/s eta 0:00:03
   ---------- ----------------------------- 5.0/18.8 MB 6.6 MB/s eta 0:00:03
   --------------- ------------------------ 7.3/18.8 MB 7.7 MB/s eta 0:00:02
   ------------------- -------------------- 9.2/18.8 MB 7.9 MB/s eta 0:00:02
   ----------------------- ---------------- 11.0/18.8 MB 8.1 MB/s eta 0:00:01
   ---------------------------- ----------- 13.4/18.8 MB 8.1 MB/s eta 0:00:01
   -------------------------------- ------- 15.2/18.8 MB 8.2 MB/s eta 0:00:01
   ------------------------------------ --- 17.0/18.8 MB 8.3 MB/s eta 0:00:01
   ---------------------------------------  18.6/18.8 MB 8.5 MB/s eta 0:00:01
   ---------------------------------------- 18.8/18.8 MB 8.1 MB/s  0:00:02


In [4]:
!pip install kagglehub

In [5]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jjinho/wikipedia-2023-07-faiss-index")

print("Path to dataset files:", path)

c:\envs\nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 5.43G/5.43G [08:29<00:00, 11.4MB/s]

Extracting files...


Path to dataset files: ~\.cache\kagglehub\datasets\jjinho\wikipedia-2023-07-faiss-index\versions\1


In [8]:
import os
from faiss import write_index, read_index
sentence_index = read_index(os.path.expanduser("~/.cache/kagglehub/datasets/jjinho/wikipedia-2023-07-faiss-index/versions/1/wikipedia_202307.index"))

In [30]:
print(f"Liczba wektorów: {sentence_index.ntotal}")
print(f"Wymiar: {sentence_index.d}")
print(f"Typ metryki: {'INNER_PRODUCT' if sentence_index.metric_type == 1 else 'L2'}")

Liczba wektorów: 6286775
Wymiar: 384
Typ metryki: INNER_PRODUCT


In [15]:
!pip install -U sentence-transformers

In [16]:
from sentence_transformers import SentenceTransformer
embeddings_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

c:\envs\nlp\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in ~\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better 

In [2]:
import os
import faiss
import numpy as np
import requests
import json
from typing import List, Dict, Any, Optional
from sentence_transformers import SentenceTransformer
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.outputs import ChatResult, ChatGeneration
from langchain_core.callbacks import CallbackManagerForLLMRun
from datetime import datetime

FAISS_INDEX_PATH = os.path.expanduser("~/.cache/kagglehub/datasets/jjinho/wikipedia-2023-07-faiss-index/versions/1/wikipedia_202307.index")
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
CLARIN_API_URL = "https://services.clarin-pl.eu/api/v1/oapi/chat/completions"
CLARIN_API_TOKEN = os.environ.get("CLARIN_API_TOKEN")  # set your own CLARIN token in the environment

c:\envs\nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
faiss_index = faiss.read_index(FAISS_INDEX_PATH)
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print(f"{faiss_index.ntotal} dokumenty")

6286775 dokumenty


In [4]:
class ClarinChatLLM(BaseChatModel):
    """Wrapper dla CLARIN Chat API - zgodny z LangChain"""
    
    model_name: str = "bielik"
    api_url: str = CLARIN_API_URL
    api_token: Optional[str] = None
    timeout: int = 60
    
    @property
    def _llm_type(self) -> str:
        return "clarin_chat_llm"
    
    def _generate(
        self,
        messages: List[HumanMessage],
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs
    ) -> ChatResult:
        # Konwersja wiadomości
        clarin_messages = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                clarin_messages.append({"role": "system", "content": msg.content})
            elif isinstance(msg, HumanMessage):
                clarin_messages.append({"role": "user", "content": msg.content})
            elif isinstance(msg, AIMessage):
                clarin_messages.append({"role": "assistant", "content": msg.content})
        
        # Payload
        payload = {
            "model": kwargs.get("model", self.model_name),
            "messages": clarin_messages,
        }
        
        headers = {
            "Authorization": f"Bearer {CLARIN_API_TOKEN}",
            "Content-Type": "application/json"
        }
        
        try:
            response = requests.post(self.api_url, headers=headers, json=payload, timeout=self.timeout)
            
            if response.status_code == 200:
                data = response.json()
                
                # Parsowanie odpowiedzi
                if "choices" in data and len(data["choices"]) > 0:
                    choice = data["choices"][0]
                    if "message" in choice and "content" in choice["message"]:
                        answer_text = choice["message"]["content"]
                    else:
                        answer_text = str(choice)
                elif "content" in data:
                    answer_text = data["content"]
                else:
                    answer_text = "Nie udało się odczytać odpowiedzi z API"
                
                # WAŻNE: Zwracamy ChatResult, nie AIMessage!
                message = AIMessage(content=answer_text.strip())
                generation = ChatGeneration(message=message)
                return ChatResult(generations=[generation])
                
            else:
                error_msg = f"API Error {response.status_code}: {response.text[:200]}"
                message = AIMessage(content=error_msg)
                generation = ChatGeneration(message=message)
                return ChatResult(generations=[generation])
                
        except Exception as e:
            error_msg = f"Error: {str(e)}"
            message = AIMessage(content=error_msg)
            generation = ChatGeneration(message=message)
            return ChatResult(generations=[generation])
    
    def _stream(self, *args, **kwargs):
        raise NotImplementedError("No streaming")

In [16]:
import pandas as pd

index_df = pd.read_parquet("wiki_2023_index.parquet")
s_articles = pd.read_parquet("s.parquet")
s_articles.head()


,id,title,text,categories
0,27173,S,"S, or s, is the nineteenth letter in the Latin...",[ISO basic Latin letters]
1,7367653,S & S Sandwich Shop,The S & S Sandwich Shop (also known as the S&S...,"[Buildings and structures in Miami, National R..."
2,43672597,"S & W Cafeteria (Asheville, North Carolina)",S & W Cafeteria is a historic S & W Cafeteria ...,[Commercial buildings on the National Register...
3,35875385,S (EP),S is the second extended play (EP) by American...,"[2013 EPs, Contemporary R&B; EPs, SZA albums]"
4,28848950,S (Los Angeles Railway),"S was a streetcar line in Los Angeles, Califor...","[Los Angeles Railway routes, Railway lines ope..."


In [38]:
index_df.head()

,id,title,text,categories,context,file
0,49495844,A & B High Performance Firearms,A & B High Performance Firearms: A & B High Pe...,"[Defunct firearms manufacturers, Defunct manuf...",A & B High Performance Firearms was a competit...,a.parquet
1,3579086,A & C Black,A & C Black: A & C Black is a British book pub...,"[Encyclopædia Britannica, Ornithological publi...",A & C Black is a British book publishing compa...,a.parquet
2,62397582,A & F Harvey Brothers,"A & F Harvey Brothers: A & F Harvey Brothers, ...",[Cotton mills],"A & F Harvey Brothers, first Spinning Cotton M...",a.parquet
3,15547032,A & G Price,A & G Price: A & G Price Limited is an enginee...,"[Locomotive manufacturers of New Zealand, Tham...",A & G Price Limited is an engineering firm and...,a.parquet
4,8021609,A & M Karagheusian,A & M Karagheusian: thumb|right|238px|A portio...,"[1904 establishments in the United States, Arm...",thumb|right|238px|A portion of the Karagheusia...,a.parquet


In [43]:
l_articles = pd.read_parquet("l.parquet")

In [79]:
def search_documents(query: str, index, embedding_model, top_k: int = 5) -> List[Dict[str, Any]]:
    """Wyszukuje dokumenty w indeksie FAISS i zwraca content zależnie od pliku"""
    
    # embedding zapytania
    query_embedding = embedding_model.encode(query)
    query_embedding = np.array([query_embedding]).astype('float32')
    
    if index.metric_type == faiss.METRIC_INNER_PRODUCT:
        faiss.normalize_L2(query_embedding)
    
    distances, indices = index.search(query_embedding, top_k)
    
    results = []
    
    for i, (score, idx) in enumerate(zip(distances[0], indices[0])):
        if idx == -1:
            continue
        
        # Pobieramy wiersz z index_df
        row_index = index_df.iloc[idx]
        doc_id = row_index['id']
        file_name = index_df.iloc[idx]["file"]
        if file_name == "s.parquet":
            # Szukamy w s_articles po id
            doc_row = s_articles[s_articles['id'] == doc_id]
            if not doc_row.empty:
                content = f"{doc_row.iloc[0]['title']}\n{doc_row.iloc[0]['text']}"
            else:
                content = f"Dokument #{doc_id} (brak w s_articles)"
        elif file_name == "l.parquet":
            doc_row = l_articles[l_articles['id'] == doc_id]
            if not doc_row.empty:
                content = f"{doc_row.iloc[0]['title']}\n{doc_row.iloc[0]['text']}"
            else:
                content = f"Dokument #{doc_id} (brak w l_articles)"
        else:
            # Bierzemy z index_df
            content = f"{row_index['title']}\n{row_index['text']}"
        
        results.append({
            "id": int(doc_id),
            "score": float(score),
            "rank": i + 1,
            "content": content
        })
    
    return results

In [80]:
run_example_queries(rag_system)


🧪 PRZYKŁADOWE ZAPYTANIA TESTOWE

📝 Test 1/10: 'Which player has won the most Super Bowls?'

🔍 Zapytanie: Which player has won the most Super Bowls?
Krok 1: Wyszukiwanie dokumentów...
   ✅ Znaleziono 3 dokumentów
   • Dok #59879929 (score: 0.668)
   • Dok #9335777 (score: 0.775)
   • Dok #3401197 (score: 0.791)
Krok 2: Przygotowanie promptu...
content List of players who have won the most NFL championships
The NFL Championships describes both the Championship games of the National Football League, which ran from 1920 to 1969; and the Super Bowl, which has run from 1966 to the present. Many players and teams have won these championships on multiple occasions, both during the NFL Championships and the Super Bowl era. The majority of these players were on the Green Bay Packers between 1960 and 1967, a period in which the Packers won three NFL Championships and the first two Super Bowl games. ==Seven championships== * Tom Brady (quarterback) won Super Bowls XXXVI, XXXVIII, XXXIX, XLIX, LI,

KeyboardInterrupt: 

In [81]:
class RAGSystem:
    def __init__(self, index, embedding_model, llm):
        self.index = index
        self.embedding_model = embedding_model
        self.llm = llm
    
    def ask(self, query: str, top_k: int = 10) -> Dict[str, Any]:
        """Pełny pipeline RAG"""
        print(f"\n🔍 Zapytanie: {query}")
        
        # Krok 1: Wyszukiwanie dokumentów
        print("Krok 1: Wyszukiwanie dokumentów...")
        documents = search_documents(query, self.index, self.embedding_model, top_k)
        
        if not documents:
            return {
                "query": query,
                "answer": "Nie znaleziono odpowiednich dokumentów.",
                "documents": [],
                "success": False
            }
        
        print(f"   ✅ Znaleziono {len(documents)} dokumentów")
        for doc in documents:
            print(f"   • Dok #{doc['id']} (score: {doc['score']:.3f})")
        
        # Krok 2: Przygotowanie promptu
        print("Krok 2: Przygotowanie promptu...")
        context = self._create_context(documents)
        prompt = self._create_prompt(query, context)
        
        # Krok 3: Generacja odpowiedzi
        print("Krok 3: Generacja odpowiedzi...")
        try:
            # Używamy invoke bezpośrednio na llm z promptem
            messages = [
                SystemMessage(content=prompt.messages[0].content),
                HumanMessage(content=prompt.messages[1].content)
            ]
            
            result = self.llm.invoke(messages)
            
            # Extract answer from ChatResult
            if hasattr(result, 'generations') and result.generations:
                answer = result.generations[0].message.content
            elif isinstance(result, ChatResult):
                answer = result.generations[0].message.content
            elif isinstance(result, AIMessage):
                answer = result.content
            else:
                answer = str(result)
            
            print(f"   ✅ Odpowiedź wygenerowana")
            
            return {
                "query": query,
                "answer": answer,
                "documents": documents,
                "success": True,
                "metadata": {
                    "documents_count": len(documents),
                    "top_score": documents[0]['score'],
                    "answer_length": len(answer)
                }
            }
            
        except Exception as e:
            error_msg = f"Błąd podczas generowania odpowiedzi: {str(e)}"
            print(f"   ❌ {error_msg}")
            
            return {
                "query": query,
                "answer": error_msg,
                "documents": documents,
                "success": False,
                "error": str(e)
            }
    
    def _create_context(self, documents: List[Dict]) -> str:
        """Tworzy kontekst z dokumentów"""
        context_parts = []
        
        for i, doc in enumerate(documents):
            context_parts.append(f"[Dokument {i+1}] {doc['content']}")
            print("content", doc["content"])
        return "\n\n".join(context_parts)
    
    def _create_prompt(self, query: str, context: str) -> ChatPromptTemplate:
        """Tworzy prompt RAG"""
        return ChatPromptTemplate.from_messages([
            SystemMessage(content="""
You are a helpful assistant. You will receive documents in English and a user question in Polish. 

IMPORTANT RULES:

- Answer ONLY in Polish.
- Base your answers ONLY on the provided documents.
- If the documents do not contain the answer, say: "Nie mogę znaleźć odpowiedzi w dostarczonych materiałach."
- Answer concisely and precisely.
- If possible, indicate which documents you used to answer.
- Translate only the parts of the documents necessary to answer the question.
- Use natural, fluent Polish.
"""),
            
            HumanMessage(content=f"""PYTANIE UŻYTKOWNIKA: {query}

DOSTARCZONE DOKUMENTY:
{context}

Proszę odpowiedzieć na pytanie na podstawie powyższych dokumentów.""")
        ])

In [109]:
def run_example_queries(rag_system):
    """Uruchamia przykładowe zapytania"""
    
    test_queries = [
    # "Which player has won the most Super Bowls?",
    # "Who is Shohei Ohtani?",
    # "Who has won more Formula 1 championships, Max Verstappen, Lewis Hamilton, or Michael Schumacher? How many has each won?",
    # "Which league does Sunderland AFC play in? What is their mascot and nickname?",
    # "Who was the MVP of Super Bowl III?",
    # "Who has the most touchdowns in the history of the San Diego Chargers?",
    "Which city is larger, San Diego or Los Angeles?",
    # "What's Silesia known for?",
    # "Who is Steve McQueen?",
    # "sports"
]
        
    # test_queries = [
    #     "Który zawodnik wygrał najwięcej Super Bowl w historii?",
    #     "Kto to Shohei Ohtani?",
    #     "Kto wygrał więcej mistrzostw w formule 1, Max Verstappen, Lewis Hamilton czy Michael Schumacher? Po ile każdy wygrał?",
    #     "W której lidze gra Sunderland AFC? Jak się nazywa ich maskotka? Jaki mają przydomek?",
    #     "Kto był MVP Super Bowl III?",
    #     "Które miasto jest większe, San Diego czy San Francisco?",
    #     "Z czego znany jest śląsk?",
    #     "Kto to Steve McQueen?"
    # ]
    print("\n" + "="*60)
    print("🧪 PRZYKŁADOWE ZAPYTANIA TESTOWE")
    print("="*60)
    
    results = []
    
    for i, query in enumerate(test_queries):
        print(f"\n📝 Test {i+1}/{len(test_queries)}: '{query}'")
        
        result = rag_system.ask(query, top_k=10)
        results.append(result)
        
        # Wyświetl podsumowanie
        print(f"\n📊 PODSUMOWANIE:")
        print(f"   Sukces: {result['success']}")
        print(f"   Dokumenty: {len(result['documents'])}")
        if result['documents']:
            print(f"   Najlepszy wynik: {result['documents'][0]['score']:.3f}")
        
        print(f"\n💡 ODPOWIEDŹ (fragment):")
        print("-"*40)
        print(result['answer'][:300] + ("..." if len(result['answer']) > 300 else ""))
        print("-"*40)
        
        # Zapisz do pliku
        timestamp = datetime.now().strftime('%H%M%S')
        filename = f"results/bielik_en_questions/rag_result_{i+1}_{timestamp}.json"
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)
        print(f"💾 Zapisano do: {filename}")
    
    return results

In [107]:
def initialize_rag_system():
    """Inicjalizuje cały system RAG"""
    print("=" * 60)
    print("🚀 INICJALIZACJA SYSTEMU RAG")
    print("=" * 60)
    
    # 1. Ładowanie FAISS
    print("\n1️⃣ Ładowanie indeksu FAISS...")
    try:
        faiss_index = faiss.read_index(FAISS_INDEX_PATH)
        print(f"   ✅ Indeks załadowany: {faiss_index.ntotal} dokumentów")
    except Exception as e:
        print(f"   ⚠️ Błąd ładowania FAISS: {e}")
        print("   Tworzę dummy index dla testów...")
        dim = 384
        faiss_index = faiss.IndexFlatIP(dim)
        # Dodaj dummy vectors
        dummy_data = np.random.randn(100, dim).astype('float32')
        faiss.normalize_L2(dummy_data)
        faiss_index.add(dummy_data)
    
    # 2. Model embeddingów
    print("\n2️⃣ Ładowanie modelu embeddingów...")
    embedding_model = SentenceTransformer(EMBEDDING_MODEL)
    print(f"   ✅ Model: {EMBEDDING_MODEL}")
    
    # 3. CLARIN LLM
    print("\n3️⃣ Inicjalizacja CLARIN LLM...")
    clarin_llm = ClarinChatLLM(
        model_name="bielik",
        max_tokens=400,
        temperature=0.2,
        timeout=45
    )
    print(f"   ✅ LLM gotowy")
    
    # 4. System RAG
    print("\n4️⃣ Tworzenie systemu RAG...")
    rag_system = RAGSystem(faiss_index, embedding_model, clarin_llm)
    print("   ✅ System RAG zainicjalizowany")
    
    return rag_system

In [108]:
rag_system = initialize_rag_system()



🚀 INICJALIZACJA SYSTEMU RAG

1️⃣ Ładowanie indeksu FAISS...
   ✅ Indeks załadowany: 6286775 dokumentów

2️⃣ Ładowanie modelu embeddingów...
   ✅ Model: sentence-transformers/all-MiniLM-L6-v2

3️⃣ Inicjalizacja CLARIN LLM...
   ✅ LLM gotowy

4️⃣ Tworzenie systemu RAG...
   ✅ System RAG zainicjalizowany


In [110]:
run_example_queries(rag_system)


🧪 PRZYKŁADOWE ZAPYTANIA TESTOWE

📝 Test 1/1: 'Which city is larger, San Diego or Los Angeles?'

🔍 Zapytanie: Which city is larger, San Diego or Los Angeles?
Krok 1: Wyszukiwanie dokumentów...
   ✅ Znaleziono 10 dokumentów
   • Dok #18110 (score: 0.677)
   • Dok #7313687 (score: 0.677)
   • Dok #44646856 (score: 0.689)
   • Dok #81605 (score: 0.700)
   • Dok #4155828 (score: 0.749)
   • Dok #40905972 (score: 0.756)
   • Dok #71812141 (score: 0.798)
   • Dok #40226337 (score: 0.803)
   • Dok #2209531 (score: 0.805)
   • Dok #26036606 (score: 0.806)
Krok 2: Przygotowanie promptu...
content Los Angeles
Los Angeles ( ; , ), often referred to by its initials L.A., is the largest city in California, the most populous U.S. state. It is the commercial, financial, and cultural center of Southern California. Los Angeles is the second-most populous city in the United States after New York City, with a population of roughly 3.9 million residents within the city limits . Los Angeles has a Mediterra

[{'query': 'Which city is larger, San Diego or Los Angeles?',
  'answer': 'Na podstawie dostępnych dokumentów nie ma bezpośredniego porównania wielkości między San Diego a Los Angeles. Jednakże można wywnioskować następujące informacje:\n\n1. Los Angeles jest znacznie większym miastem niż San Diego, zarówno jeśli chodzi o liczbę mieszkańców, jak i obszar geograficzny.\n\n2. Los Angeles jest ósmym co do wielkości miastem w Stanach Zjednoczonych, podczas gdy San Diego zajmuje pozycję w pierwszej dziesiątce największych miast Kalifornii.\n\n3. Los Angeles ma znacznie większą populację niż San Diego. Według dokumentu [Dokument 1], Los Angeles jest drugim co do wielkości miastem w USA po Nowym Jorku, z populacją około 3,9 miliona mieszkańców w granicach miasta.\n\n4. San Diego nie posiada tak rozbudowanej infrastruktury miejskiej jak Los Angeles, choć jest ważnym ośrodkiem gospodarczym i kulturalnym w Kalifornii.\n\nPodsumowując, na podstawie dostępnych informacji można stwierdzić, że Los A

In [31]:
s_articles[s_articles["id"]=="37426141"]

,id,title,text,categories
211039,37426141,Shohei Ohtani,", nicknamed ""Shotime"" and “The Unicorn”, is a ...","[1994 births, Living people, American League A..."


In [28]:
index_df.iloc[5200169]

id                                                     37426141
title                                             Shohei Ohtani
text          Shohei Ohtani: , nicknamed "Shotime" and “The ...
categories    [1994 births, Living people, American League A...
context       , nicknamed "Shotime" and “The Unicorn”, is a ...
file                                                  s.parquet
Name: 5200169, dtype: object

In [75]:
import requests

CLARIN_API_URL = "https://services.clarin-pl.eu/api/v1/oapi/chat/completions"
CLARIN_API_TOKEN = "<REDACTED>"  # Tutaj wklej swój token

print("\n🧪 Testowanie połączenia z CLARIN API...")
try:
    test_payload = {
        "model": "bielik",
        "messages": [{"role": "user", "content": "Test"}],
        "max_tokens": 10
    }
    
    headers = {
        "Authorization": f"Bearer {CLARIN_API_TOKEN}",
        "Content-Type": "application/json"
    }
    
    response = requests.post(
        CLARIN_API_URL, 
        json=test_payload, 
        headers=headers,  # Dodaj nagłówki z tokenem
        timeout=10
    )
    
    print(f"   Status API: {response.status_code}")
    if response.status_code != 200:
        print(f"   ⚠️ Uwaga: API zwróciło status {response.status_code}")
        print(f"   Odpowiedź: {response.text}")
except Exception as e:
    print(f"   ⚠️ Nie można połączyć z API: {e}")


🧪 Testowanie połączenia z CLARIN API...
   Status API: 200


In [62]:
response.text

'{"choices":[{"index":0,"message":{"role":"assistant","content":"Hello! How can I help you today? If"},"logprobs":null}],"created":1765451291,"object":"chat.completion","usage":{"completion_tokens":10,"prompt_tokens":11,"total_tokens":21},"timing":{"duration":0.22229838371276855,"speed":44.984582582125235},"model":"speakleash/Bielik-11B-v2.2-Instruct","time":0.2667806148529053}'

### porownanie

In [113]:
import os
import json

indices = [1, 2, 3, 4, 5, 7, 8, 9, 10]
models = [
    "results/bielik_en_questions",
    "results/llama_en_questions",
    "results/pllum_en_questions"
]

for i in indices:
    print(f"\n===== rag_result_{i} =====")
    for model in models:
        for file in os.listdir(model):
            if file.startswith(f"rag_result_{i}_"):
                path = os.path.join(model, file)

                # otwarcie i wczytanie JSON
                with open(path, "r", encoding="utf-8") as f:
                    data = json.load(f)

                # wypisanie nazwy modelu i odpowiedzi
                print(f"\nModel: {model}")
                print(f"Question: {data.get('query','')}")
                print(f"Answer: {data.get('answer', 'BRAK POLA answer')}")




===== rag_result_1 =====

Model: results/bielik_en_questions
Question: Which player has won the most Super Bowls?
Answer: 

Model: results/llama_en_questions
Question: Which player has won the most Super Bowls?
Answer: Tom Brady jest graczem, który wygrał najwięcej Super Bowlów. Wygrał ich siedem.

Model: results/pllum_en_questions
Question: Which player has won the most Super Bowls?
Answer: Odpowiedź: 6. Lista najbardziej utytułowanych zawodników w Super Bowl..

===== rag_result_2 =====

Model: results/bielik_en_questions
Question: Who is Shohei Ohtani?
Answer: Shohei Ohtani jest japońskim zawodnikiem baseballu, który gra jako miotacz i jednocześnie pełni funkcję uderzającego. Urodził się 5 lipca 1994 roku w Mizusawa, obecnie części miasta Ōshū w prefekturze Iwate w Japonii.

Ohtani rozpoczął swoją karierę profesjonalną w 2013 roku w Nippon Professional Baseball (NPB) w drużynie Hokkaido Nippon-Ham Fighters. Grał tam przez pięć lat, zdobywając wiele nagród, w tym nagrodę dla najlepsz

Pytanie 1: LLama ok, bielik, pllum nie

Pytanie 2: Bielik fajnie, długo, na temat, streszczenie kariery. Llama zwięźle, spoko. Pllum przekopiował wikipedie

Pytanie 3: Bielik - Schumacher ma 7 zwycięstw! Llama: świetnie, Pllum: pogubił sie, Hamilton ma 103 zwycięstwa, nie 41, losowe kraje

Pytanie 4: Bielik - dobrze odpowiedział i mnie przeprosił, Llama, zwięźle na temat. Pllum miał wylew (autor tego zdania to Bartosz Tramś)

Pytanie 5: Bielik - nie, Llama - dobrze, pllum - co?

Pytanie 7: trick question, dzięki że mówicie że nie wiecie

Pytanie 8: Bielik, Llama dobrze, Pllum kocha Adama Małysza

Pytanie 9: Król chłodu, -==-

Pytanie 10: Bielik - nie wie ale sie domyśla, Llama dobrze zgadł, Pllum dobrze!~!!!!!